In [1]:
import torch
import numpy as np

In [40]:
torch.manual_seed(42)
F = torch.rand((2,2,3))
print(F)

tensor([[[0.8823, 0.9150, 0.3829],
         [0.9593, 0.3904, 0.6009]],

        [[0.2566, 0.7936, 0.9408],
         [0.1332, 0.9346, 0.5936]]])


In [43]:
trial = 1
Z_temp = F@F.mT
torch.trace(Z_temp[trial])**2 - torch.trace(Z_temp[trial]@Z_temp[trial])

tensor(0.3706)

In [42]:
Z2 = torch.einsum("knt,lmt->klnm",F, F) # (K, K, N, N) -- K^2 NxN matrices 
trace_1 = torch.einsum("klnn->kl", Z2)
Z_squared = torch.einsum("klnm,klmj->klnj",Z2,Z2) # (K, K, N, N)
trace_2 = torch.einsum("klnn->kl",Z_squared)
trace_1**2 - trace_2

tensor([[ 0.9424, -0.4610],
        [-0.4610,  0.3706]])

In [44]:
(trace_1 ** 2 - trace_2).sum()

tensor(0.3909)

In [46]:
from nlb_tools.nwb_interface import NWBDataset
path = '/Users/omomalley03/Documents/Dissertation/Data/000128/sub-Jenkins/sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb'
ds = NWBDataset(path)
print('native bin_width:', ds.bin_width, 'ms')
print('shape:', ds.data.shape)
print('signal groups:', ds.data.columns.get_level_values(0).unique().tolist())
print()
print('trial_info cols:', list(ds.trial_info.columns))
print('n trials:', len(ds.trial_info))
print()
ti = ds.trial_info
for c in ['trial_type','trial_version','split','maze_id','num_targets','num_barriers']:
    if c in ti.columns:
        u = ti[c].dropna().unique()
        print(f'  {c}: nunique={ti[c].nunique()}  sample={u[:8]}  dtype={ti[c].dtype}')
print()
print('=== first 3 rows (narrow cols) ===')
keep = [c for c in ti.columns if c not in ('target_pos',)]
print(ti[keep].head(3).to_string())
print()
print('=== target_pos sample ===')
if 'target_pos' in ti.columns:
    print(repr(ti['target_pos'].iloc[0]))


native bin_width: 1 ms
shape: (6952301, 190)
signal groups: ['cursor_pos', 'eye_pos', 'hand_pos', 'hand_vel', 'heldout_spikes', 'spikes']

trial_info cols: ['trial_id', 'start_time', 'end_time', 'trial_type', 'trial_version', 'maze_id', 'success', 'target_on_time', 'go_cue_time', 'move_onset_time', 'rt', 'delay', 'num_targets', 'target_pos', 'num_barriers', 'barrier_pos', 'active_target', 'split']
n trials: 2295

  trial_type: nunique=36  sample=[25  3 22 29 21  2 16 23]  dtype=int64
  trial_version: nunique=3  sample=[2 1 0]  dtype=int64
  split: nunique=2  sample=['val' 'train']  dtype=object
  maze_id: nunique=36  sample=[ 84   3  66 100  65   2  76  67]  dtype=int64
  num_targets: nunique=2  sample=[3 1]  dtype=int64
  num_barriers: nunique=5  sample=[8 6 9 0 7]  dtype=int64

=== first 3 rows (narrow cols) ===
   trial_id             start_time               end_time  trial_type  trial_version  maze_id  success         target_on_time            go_cue_time        move_onset_time   

In [ ]:
Z = torch.einsum("knt,lmt->klnm",F, F) # (K, K, N, N) -- K^2 NxN matrices 
# trace_1[k,l] — trace of Z[k,l]

trace_1 = torch.einsum("klnn->kl", Z)        # (K, K)

# trace_2[k,l] - trace of Z^2
# trace_2 = torch.einsum("kim,ljm,kjn,lin->kl", F, F, F, F)  # (K, K)
Z_squared = torch.einsum("klnm,klmj->klnj",Z,Z) # (K, K, N, N)
trace_2 = torch.einsum("klnn->kl",Z_squared)

return (trace_1 ** 2 - trace_2).sum(), (trace_1 ** 2 + trace_2).sum()



In [8]:
Z

tensor([[[1.7622, 1.4337],
         [1.4337, 1.4338]]])